### First Step : Clean raw data 

The dataset has been downloaded from the website ... . Some columns are completely useless (bookmakers infos) and some rows are full of missing values. The first step is to clean the dataset.

In [1]:
import pandas as pd
from Data.data_cleaning import Datacleaner

In [ ]:
cleaner = Datacleaner("./Data/ligue1_2010_2025.csv")

cleaner = (cleaner.drop_useless_columns()
           .drop_rows(row_idxs=[1520,2607,2281])
           .fillNA(column='Div',value='F1')
           .fillNA(column='Time', value='00:00')
           .add_season())

cleaner.save_data("Data/Dataset_clean.csv")

In [2]:
df = pd.read_csv("./Data/Dataset_clean.csv")

print(df)


     Div        Date   Time    HomeTeam      AwayTeam  FTHG  FTAG FTR  HTHG  \
0     F1    07/08/10  00:00     Auxerre       Lorient   2.0   2.0   D   1.0   
1     F1    07/08/10  00:00        Lens         Nancy   1.0   2.0   A   0.0   
2     F1    07/08/10  00:00        Lyon        Monaco   0.0   0.0   D   0.0   
3     F1    07/08/10  00:00   Marseille          Caen   1.0   2.0   A   0.0   
4     F1    07/08/10  00:00        Nice  Valenciennes   0.0   0.0   D   0.0   
...   ..         ...    ...         ...           ...   ...   ...  ..   ...   
5589  F1  14/12/2025  14:00        Lyon      Le Havre   1.0   0.0   H   0.0   
5590  F1  14/12/2025  16:15     Auxerre         Lille   3.0   4.0   A   0.0   
5591  F1  14/12/2025  16:15        Lens          Nice   2.0   0.0   H   1.0   
5592  F1  14/12/2025  16:15  Strasbourg       Lorient   0.0   0.0   D   0.0   
5593  F1  14/12/2025  19:45   Marseille        Monaco   1.0   0.0   H   0.0   

      HTAG  ...  AST    HC   AC    HF    AF   HY   

### Second step : Classical Bradley-Terry

In [3]:
from Data.data_processing import DataProcesser

In [4]:
processer = DataProcesser("./Data/Dataset_clean.csv")

processer = processer.filter_year("10-11")

data = processer.get_data()

teams = data['Teams']
W_matrix = data['Victory Matrix']
D_matrix = data['Draw Matrix']


In [5]:
from BradleyTerry_classical.BT_classical import BradleyTerry

In [6]:
bt_model = BradleyTerry(lambda_draw=-1, learning_rate=0.01, n_iterations=1000)
bt_model.fit(W=W_matrix, D=D_matrix, teams=teams)

strength = bt_model.predict_strength()

Tolerance threshold attained after 276 iterations


In [7]:
result = {team: float(value) for team,value in zip(bt_model.teams,strength)}
result

{'Arles': -1.4487725118094927,
 'Auxerre': 0.06476223308424217,
 'Bordeaux': 0.06476223308424217,
 'Brest': -0.1829597869120999,
 'Caen': -0.18295978691209988,
 'Lens': -0.6297688512124878,
 'Lille': 1.1270349039110978,
 'Lorient': -0.05891934660309426,
 'Lyon': 0.5691025624886928,
 'Marseille': 0.7687544628504622,
 'Monaco': -0.1829597869120999,
 'Montpellier': -0.1829597869120999,
 'Nancy': -0.18295978691209988,
 'Nice': -0.18295978691209985,
 'Paris SG': 0.44018772122289984,
 'Rennes': 0.18873145155178808,
 'Sochaux': 0.18873145155178805,
 'St Etienne': -0.05891934660309426,
 'Toulouse': -0.12085412281952175,
 'Valenciennes': 0.002925880775077108}

In [8]:
from Test.test_BT_classical import BT_classical_TEST

In [10]:
test = BT_classical_TEST(bt_model)

test.run_all()



CONVERGENCE CHECK
Gradient Norm: 9.841550e-07
Max gradient component (abs. value): 9.304496e-07
Tolerance: 1e-06
You reached CV


LOG-LIKELIHOOD CHECK
Log-likelihood at the optimum: -439.0711

Random points (n=20):
    - Max: -483.9733
    - Mean: -519.5993
    - Min: -560.4442

Difference between optimum loglik - best random loglik: 44.9022
The optimum is better than any random point !

Perturbated optimum (n=200):
    - Max: -439.3621
    - Mean: -439.7936
    - Min: -440.9550

Difference between optimum loglik - best perturbated loglik: 0.2910
The optimum is better than any perturbated point !


PREDICTIONS QUALITY CHECK
Number of analyzed pairs: 380
Total outcomes analyzed: 1140

--- Global Metrics (all 3 outcomes) ---
    - MAE (Mean Absolute Error): 0.2864
    - RMSE (Root Mean Squared Error): 0.3444
    - Correlation pred/obs: 0.2082

--- Metrics per Outcome ---
Win:  MAE = 0.2699, Corr = 0.4082
Draw: MAE = 0.3194, Corr = 0.0854
Loss: MAE = 0.2699, Corr = 0.4082


RATING COHER